In [ ]:
import pandas as pd
from pathlib import Path
import re
from datetime import datetime
from tqdm import tqdm
from langdetect import detect
from deep_translator import GoogleTranslator
from ftfy import fix_text
import unicodedata

In [ ]:
# normalise the titles: lowercase everything + strip leading and trailing spaces

def normalize_columns(df):
    new_columns = {col: col.lower().strip() for col in df.columns}
    df.rename(columns=new_columns, inplace=True)
    if "title" in df.columns:
        df["title"] = df["title"].astype(str).str.lower().str.strip()
        
    return df

In [ ]:
# merge function

def load_files(folder_path):
    folder = Path(folder_path)
    dfs = []

    for file_path in folder.glob("*.xlsx"):
        df = pd.read_excel(file_path)
        df = normalize_columns(df)
        dfs.append(df)

    df_concat = pd.concat(dfs, ignore_index=True)
    return df_concat

In [ ]:
folder_path = "../2_scraped_files"  # modify
df_final = load_files(folder_path)

In [ ]:
# clean non-standard punctuation

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = fix_text(text) # fix odd characters
    text = re.sub(r'\s+', ' ', text) # replace runs of spaces with a single one
    text = re.sub(r"[“”«»]", '"', text) # standardise quotation marks
    text = re.sub(r"[-–—]+", "-", text) # standardise long dashes
    text = re.sub(r"\s+([.,!?;:])", r"\1", text) # remove spaces before punctuation
    text = re.sub(r"([.,!?;:])([^\s])", r"\1 \2",text) # add a space after punctuation
    return text.strip()

df_final['title'] = df_final['title'].apply(clean_text)

In [ ]:
# format dates as dd-mm-yyyy

def format_date(date_str):
    if pd.isna(date_str):
        return None
    for fmt in ("%d/%m/%Y", "%d-%m-%Y"):
        try:
            return datetime.strptime(date_str, fmt).strftime("%d-%m-%Y")
        except:
            continue
    return None 


df_final["publication_date"] = df_final["publication_date"].apply(format_date)
df_final["expiration_date"] = df_final["expiration_date"].apply(format_date)

In [ ]:
df_final.to_excel("data.xlsx", index= False)